# TGATE Experiment

In [ ]:
!pip install -q tgate
!pip install -q transformers torch torchvision pillow accelerate huggingface-hub requests DeepCache torch-fidelity torchmetrics calflops

In [ ]:
import os
import json
import time
import torch
import numpy as np
from tqdm import tqdm
from diffusers import StableDiffusionPipeline, DiffusionPipeline
from tgate import TgateSDLoader
from torch_fidelity import calculate_metrics
from PIL import Image
import torch.nn.functional as F
from transformers import CLIPProcessor, CLIPModel
from calflops import calculate_flops

!pip freeze > requirements.txt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16 if DEVICE == 'cuda' else torch.float32
print(f'Device: {DEVICE}')

BATCH_SIZE  = 8 #standardized with other methods
NUM_STEPS   = 25 #same as baseline
NUM_IMAGES = 1000 #important for stable FID
GATE_STEP   = 10 #stop recompute cross-attention
OUTPUT_ROOT = 'outputs'
COCO_DIR    = 'coco_real'

In [ ]:
!wget -q http://images.cocodataset.org/zips/val2017.zip
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q val2017.zip
!unzip -q annotations_trainval2017.zip

In [ ]:
COCO_IMG_DIR  = 'val2017'
COCO_ANN_FILE = 'annotations/captions_val2017.json'

def load_coco_val2017():
    with open(COCO_ANN_FILE, 'r') as f:
        data = json.load(f)

    # Map image_id -> captions
    captions_dict = {}
    for ann in data['annotations']:
        img_id = ann['image_id']
        captions_dict.setdefault(img_id, []).append(ann['caption'])

    # Map image_id -> filename
    id_to_file = {img['id']: img['file_name'] for img in data['images']}
    samples = []
    for img_id, file_name in id_to_file.items():
        path = os.path.join(COCO_IMG_DIR, file_name)
        captions = captions_dict.get(img_id, [])

        if os.path.exists(path) and captions:
            samples.append({'image_path': path, 'caption': captions[0]}) # pick first caption

    return samples

samples = load_coco_val2017() #compare all dataset
print(f'Loaded {len(samples)} samples')

In [ ]:
os.makedirs(COCO_DIR, exist_ok=True)
for i, sample in enumerate(samples):
    img = Image.open(sample['image_path']).convert('RGB')
    img = img.resize((512, 512))
    img.save(f'{COCO_DIR}/{i}.png')

In [ ]:
prompts = [s['caption'] for s in samples[:NUM_IMAGES]]

def load_pipeline_tgate(name):
    if name == 'sd1.5':
        pipe = StableDiffusionPipeline.from_pretrained(
            'runwayml/stable-diffusion-v1-5', torch_dtype=DTYPE)
    elif name == 'sd2.1':
        pipe = DiffusionPipeline.from_pretrained(
            'sd2-community/stable-diffusion-2-1', torch_dtype=DTYPE)
    pipe = TgateSDLoader(pipe, gate_step=GATE_STEP,
                         num_inference_steps=NUM_STEPS).to(DEVICE)
    pipe.set_progress_bar_config(disable=True)
    return pipe

def load_pipeline_baseline(name):
    if name == 'sd1.5':
        pipe = StableDiffusionPipeline.from_pretrained(
            'runwayml/stable-diffusion-v1-5', torch_dtype=DTYPE)
    elif name == 'sd2.1':
        pipe = DiffusionPipeline.from_pretrained(
            'sd2-community/stable-diffusion-2-1', torch_dtype=DTYPE)
    pipe = pipe.to(DEVICE)
    pipe.set_progress_bar_config(disable=True)
    return pipe

print(f'GATE_STEP={GATE_STEP}, NUM_STEPS={NUM_STEPS}, BATCH_SIZE={BATCH_SIZE}')

In [ ]:
def generate_images_tgate(pipe, prompts, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    all_times  = []
    peak_mem   = 0.0
    total_imgs = 0

    for i in tqdm(range(0, len(prompts), BATCH_SIZE)):
        batch_prompts = prompts[i:i+BATCH_SIZE]
        batch_size    = len(batch_prompts)
        latents = torch.randn(
            (batch_size, pipe.unet.in_channels, 64, 64),
            device=DEVICE, dtype=DTYPE)

        torch.cuda.reset_peak_memory_stats()

        start  = time.time()

        images = pipe.tgate(
            batch_prompts,
            gate_step=GATE_STEP,
            num_inference_steps=NUM_STEPS,
            latents=latents
        ).images

        elapsed = time.time() - start

        all_times.append(elapsed)
        peak_mem    = max(peak_mem, torch.cuda.max_memory_allocated() / 1e9)
        total_imgs += batch_size

        for j, img in enumerate(images):
            img.save(f'{out_dir}/{i+j}.png')

    mean_t         = np.mean(all_times)
    std_t          = np.std(all_times)
    time_per_image = mean_t / BATCH_SIZE
    throughput     = total_imgs / sum(all_times)
    return mean_t, std_t, peak_mem, time_per_image, throughput


def generate_images_baseline(pipe, prompts, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    all_times  = []
    peak_mem   = 0.0
    total_imgs = 0

    for i in tqdm(range(0, len(prompts), BATCH_SIZE)):
        batch_prompts = prompts[i:i+BATCH_SIZE]
        batch_size    = len(batch_prompts)
        latents = torch.randn(
            (batch_size, pipe.unet.in_channels, 64, 64),
            device=DEVICE, dtype=DTYPE)

        torch.cuda.reset_peak_memory_stats()
        start  = time.time()
        images = pipe(
            batch_prompts,
            num_inference_steps=NUM_STEPS,
            latents=latents
        ).images
        elapsed = time.time() - start

        all_times.append(elapsed)
        peak_mem    = max(peak_mem, torch.cuda.max_memory_allocated() / 1e9)
        total_imgs += batch_size

        for j, img in enumerate(images):
            img.save(f'{out_dir}/{i+j}.png')

    mean_t         = np.mean(all_times)
    std_t          = np.std(all_times)
    time_per_image = mean_t / BATCH_SIZE
    throughput     = total_imgs / sum(all_times)

    # return time and memory metrics
    return mean_t, std_t, peak_mem, time_per_image, throughput


def compute_fid(real_dir, fake_dir):
    metrics = calculate_metrics(
        input1=real_dir, input2=fake_dir,
        fid=True, isc=False, kid=False,
        cuda=(DEVICE == 'cuda'))
    return metrics['frechet_inception_distance']

print('Functions defined')

## MACs

In [ ]:
#measured MACs before realizing that baseline and TGATE will report identical
#because TGATE does not remove any opertations
def measure_macs(pipe, encoder_dim=768):
    try:
        dummy_sample         = torch.randn(1, 4, 64, 64, device=DEVICE, dtype=DTYPE)
        dummy_timestep       = torch.tensor([1], device=DEVICE)
        dummy_encoder_hidden = torch.randn(1, 77, encoder_dim, device=DEVICE, dtype=DTYPE)

        flops, macs, params = calculate_flops(
            model=pipe.unet,
            kwargs={
                'sample':                dummy_sample,
                'timestep':              dummy_timestep,
                'encoder_hidden_states': dummy_encoder_hidden
            },
            output_as_string=True,
            output_precision=4
        )
        return macs
    except Exception as e:
        return f'Error: {e}'

print('Measuring MACs for SD1.5 baseline...')
pipe_tmp = load_pipeline_baseline('sd1.5')
macs_sd15_base = measure_macs(pipe_tmp, encoder_dim=768)
del pipe_tmp; torch.cuda.empty_cache()
print(f'  SD1.5 baseline MACs: {macs_sd15_base}')

print('Measuring MACs for SD1.5 + TGATE...')
pipe_tmp = load_pipeline_tgate('sd1.5')
macs_sd15_tgate = measure_macs(pipe_tmp, encoder_dim=768)
del pipe_tmp; torch.cuda.empty_cache()
print(f'  SD1.5 TGATE MACs: {macs_sd15_tgate}')

print('Measuring MACs for SD2.1 baseline...')
pipe_tmp = load_pipeline_baseline('sd2.1')
macs_sd21_base = measure_macs(pipe_tmp, encoder_dim=1024)
del pipe_tmp; torch.cuda.empty_cache()
print(f'  SD2.1 baseline MACs: {macs_sd21_base}')

print('Measuring MACs for SD2.1 + TGATE...')
pipe_tmp = load_pipeline_tgate('sd2.1')
macs_sd21_tgate = measure_macs(pipe_tmp, encoder_dim=1024)
del pipe_tmp; torch.cuda.empty_cache()
print(f'  SD2.1 TGATE MACs: {macs_sd21_tgate}')

## Run All Experiments

In [ ]:
configs = [
    ('sd1.5', False),
    ('sd1.5', True),
    ('sd2.1', False),
    ('sd2.1', True),
]

results = {}

for model, use_tgate in configs:
    print(f'\nRunning {model} | tgate={use_tgate}')
    total_start = time.time()

    if use_tgate:
        pipe    = load_pipeline_tgate(model)
        out_dir = f'{OUTPUT_ROOT}/{model}_tgate'
        mean_t, std_t, peak_mem, tpi, throughput = generate_images_tgate(pipe, prompts, out_dir)
    else:
        pipe    = load_pipeline_baseline(model)
        out_dir = f'{OUTPUT_ROOT}/{model}_base'
        mean_t, std_t, peak_mem, tpi, throughput = generate_images_baseline(pipe, prompts, out_dir)

    total_elapsed = time.time() - total_start
    hrs  = int(total_elapsed // 3600)
    mins = int((total_elapsed % 3600) // 60)
    secs = int(total_elapsed % 60)

    results[(model, use_tgate)] = {
        'dir':          out_dir,
        'time_mean':    mean_t,
        'time_std':     std_t,
        'time_per_img': tpi,
        'throughput':   throughput,
        'peak_mem_gb':  peak_mem,
        'total_time':   f'{hrs} hrs {mins} mins {secs} secs',
    }
    print(f'  Total: {hrs}h {mins}m {secs}s')
    print(f'  Time/image: {tpi:.3f}s | Throughput: {throughput:.2f} img/s | Peak VRAM: {peak_mem:.2f} GB')

    del pipe
    torch.cuda.empty_cache()

## FID

In [ ]:
fid_results = {}
for model in ['sd1.5', 'sd2.1']:
    base_dir  = results[(model, False)]['dir']
    tgate_dir = results[(model, True)]['dir']
    print(f'Computing FID for {model}...')
    fid_base  = compute_fid(COCO_DIR, base_dir)
    fid_tgate = compute_fid(COCO_DIR, tgate_dir)
    fid_results[model] = {'base': fid_base, 'tgate': fid_tgate, 'delta': fid_tgate - fid_base}
    print(f'  Base: {fid_base:.3f} | TGATE: {fid_tgate:.3f} | Δ: {fid_tgate - fid_base:.3f}')

## CLIP Score

In [ ]:
def load_clip():
    model     = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE)
    processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
    model.eval()
    return model, processor

def compute_clip_score(image_dir, prompts, model, processor):
    image_files = sorted(os.listdir(image_dir))
    scores = []
    for i in tqdm(range(0, len(image_files), BATCH_SIZE)):
        batch_files  = image_files[i:i+BATCH_SIZE]
        batch_images = [Image.open(os.path.join(image_dir, f)).convert('RGB') for f in batch_files]
        batch_texts  = [prompts[i+j] for j in range(len(batch_files))]
        inputs = processor(
            text=batch_texts, images=batch_images,
            return_tensors='pt', padding=True,
            truncation=True, max_length=77
        ).to(DEVICE)
        with torch.no_grad():
            out          = model(**inputs)
            img_embeds   = F.normalize(out.image_embeds, dim=-1)
            text_embeds  = F.normalize(out.text_embeds,  dim=-1)
            batch_scores = (img_embeds * text_embeds).sum(dim=-1)
        scores.extend(batch_scores.cpu().numpy())
    return float(np.mean(scores))

clip_model, clip_processor = load_clip()
clip_results = {}
for model in ['sd1.5', 'sd2.1']:
    base_dir  = results[(model, False)]['dir']
    tgate_dir = results[(model, True)]['dir']
    print(f'\nComputing CLIP for {model}...')
    clip_base  = compute_clip_score(base_dir,  prompts, clip_model, clip_processor)
    clip_tgate = compute_clip_score(tgate_dir, prompts, clip_model, clip_processor)
    clip_results[model] = {'base': clip_base, 'tgate': clip_tgate, 'delta': clip_tgate - clip_base}
    print(f'  Base: {clip_base:.4f} | TGATE: {clip_tgate:.4f} | Δ: {clip_tgate - clip_base:.4f}')

## Final Results

In [ ]:
print(f'GATE_STEP={GATE_STEP}, NUM_STEPS={NUM_STEPS}, BATCH_SIZE={BATCH_SIZE}')
print('='*105)
print(f'{"Model":<22} {"Total Time":<22} {"Time/Img(s)":<13} {"Var(s)":<10} {"Throughput":<13} {"Peak VRAM(GB)":<15} {"FID":<10} {"CLIP"}')
print('='*105)

for model in ['sd1.5', 'sd2.1']:
    for use_tgate in [False, True]:
        label = f'{model}, TGATE={use_tgate}'
        r     = results[(model, use_tgate)]
        fid   = fid_results[model]['base']  if not use_tgate else fid_results[model]['tgate']
        clip  = clip_results[model]['base'] if not use_tgate else clip_results[model]['tgate']
        print(f'{label:<22} {r["total_time"]:<22} {r["time_per_img"]:<13.3f} '
              f'{r["time_std"]:<10.3f} {r["throughput"]:<13.2f} '
              f'{r["peak_mem_gb"]:<15.2f} {fid:<10.3f} {clip:.4f}')

print('='*105)
print('\nSpeedup summary:')
for model in ['sd1.5', 'sd2.1']:
    base_t  = results[(model, False)]['time_mean']
    tgate_t = results[(model, True)]['time_mean']
    speedup = base_t / tgate_t
    print(f'  {model}: {speedup:.2f}x speedup | '
          f'ΔFID={fid_results[model]["delta"]:.3f} | '
          f'ΔCLIP={clip_results[model]["delta"]:.4f}')

print(f'\nMACs (SD1.5):')
print(f'  Baseline: {macs_sd15_base}')
print(f'  TGATE:    {macs_sd15_tgate}')